# Laboratorio 19. Proyecto integrador: clasificación de señales

**Pregunta guía:** ¿Cómo diseñar un flujo de Deep Learning completo sin fuga de información?

## Objetivos
Al finalizar podrás formular problema y baseline; separar train/validación/test; ajustar normalización solo con train; entrenar CNN 1D; evaluar métricas y errores.

## Prerrequisitos y conexión
Integra tensores, optimización, regularización y modelos de secuencia de 15–18. Se requiere Python básico; cada notebook puede ejecutarse de forma independiente desde un kernel limpio.

## Teoría breve
Un experimento requiere protocolo antes del modelo. Las estadísticas de preprocesamiento se estiman en train; validación selecciona configuración y test se consulta al final.

## Problema y datos
Clasificamos señales sintéticas de tres frecuencias con perturbaciones, amplitud y fase variables. Todos los datos se generan en el notebook: no se requieren descargas ni archivos locales.

## Experimento conceptual
¿La CNN 1D supera el baseline en test y qué errores quedan por explicar? Fija la semilla, ejecuta las celdas en orden y cambia una decisión cada vez.


### Preparación del entorno
Importamos NumPy y PyTorch, fijamos semillas y elegimos CPU o GPU de forma opcional.


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch import nn
import torch.nn.functional as F

SEED = 2026
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams['figure.figsize'] = (7, 4)
print('PyTorch:', torch.__version__, '| dispositivo:', device)


**Resultado e interpretación.** La celda confirma el entorno utilizado. El resto del laboratorio está diseñado para CPU; una GPU disponible se utiliza sin ser requisito.


### Definir tarea, splits y transformación
Generamos señales, fijamos particiones y calculamos normalización exclusivamente en entrenamiento.


In [ ]:
signals, labels = [], []
time = np.linspace(0, 1, 64, dtype=np.float32)
for index in range(720):
    label = index % 3; frequency = [2, 5, 9][label]
    signal = np.sin(2*np.pi*frequency*time + np.random.uniform(0, 2*np.pi))
    signal = np.random.uniform(.7, 1.3)*signal + np.random.normal(0, .25, len(time))
    signals.append(signal.astype(np.float32)); labels.append(label)
X = torch.tensor(np.array(signals), device=device).unsqueeze(1)
y = torch.tensor(labels, dtype=torch.long, device=device)
order = torch.randperm(len(y), device=device); train_idx, val_idx, test_idx = order[:500], order[500:610], order[610:]
train_mean = X[train_idx].mean(); train_std = X[train_idx].std().clamp_min(1e-6)
X = (X - train_mean) / train_std
print('shapes:', X.shape, '| splits:', len(train_idx), len(val_idx), len(test_idx))
print('baseline test:', (y[test_idx] == y[train_idx].bincount().argmax()).float().mean().item())


**Resultado e interpretación.** La normalización usa estadísticas de entrenamiento aunque se aplique luego a todos los splits. El baseline no aprende la forma temporal.


### Entrenar, evaluar y examinar errores
Entrenamos CNN 1D con AdamW y dropout; test se usa una vez para métricas y matriz de confusión.


In [ ]:
class SignalCNN(nn.Module):
    def __init__(self):
        super().__init__(); self.features = nn.Sequential(nn.Conv1d(1, 12, 5, padding=2), nn.ReLU(), nn.MaxPool1d(2), nn.Conv1d(12, 24, 5, padding=2), nn.ReLU(), nn.AdaptiveAvgPool1d(1))
        self.dropout = nn.Dropout(.2); self.output = nn.Linear(24, 3)
    def forward(self, inputs): return self.output(self.dropout(self.features(inputs).squeeze(-1)))
model = SignalCNN().to(device); optimizer = torch.optim.AdamW(model.parameters(), lr=.008, weight_decay=.01)
train_losses, val_accuracies = [], []
for epoch in range(22):
    model.train(); loss = F.cross_entropy(model(X[train_idx]), y[train_idx])
    optimizer.zero_grad(); loss.backward(); optimizer.step(); train_losses.append(loss.item())
    model.eval()
    with torch.no_grad(): val_accuracies.append((model(X[val_idx]).argmax(1) == y[val_idx]).float().mean().item())
model.eval()
with torch.no_grad(): prediction = model(X[test_idx]).argmax(1)
accuracy = (prediction == y[test_idx]).float().mean().item()
confusion = torch.bincount(y[test_idx] * 3 + prediction, minlength=9).reshape(3, 3).cpu().numpy()
print('test accuracy:', accuracy, '| etiquetas:', np.bincount(y[test_idx].cpu().numpy()))
print('matriz de confusión (filas reales):
', confusion)
fig, axes = plt.subplots(1, 2, figsize=(8, 3)); axes[0].plot(train_losses); axes[0].set_title('Pérdida train'); axes[1].plot(val_accuracies); axes[1].set_title('Exactitud validación'); plt.tight_layout(); plt.show()
wrong = torch.where(prediction != y[test_idx])[0]
print('errores test:', len(wrong), '| primeros índices dentro del test:', wrong[:10].cpu().tolist())


**Resultado e interpretación.** Interpreta matriz de confusión, baseline, curva de validación y errores concretos. El test aproxima solo la distribución sintética usada y no valida un despliegue real.


## Limitaciones y conclusiones clave
Los resultados dependen de esta semilla, tamaño de muestra, arquitectura y protocolo. No extrapoles métricas sintéticas a datos reales; separa decisiones de modelado de conclusiones generales. La celda de salida aporta la evidencia numérica y visual para responder la pregunta guía.

- Los parámetros y activaciones determinan cómo se representa la señal.
- La pérdida, el optimizador y el protocolo de evaluación forman parte del experimento.
- Una métrica aislada no describe todos los errores ni garantiza generalización.

## Ejercicios progresivos
1. **Guiado:** Guiado: visualiza una señal mal clasificada con etiqueta real y predicha.
2. **Independiente:** Independiente: compara CNN 1D con un baseline de características frecuencia/amplitud definido sin mirar test.
3. **Desafío:** Desafío: plantea una validación por grupos o por dominio que mida robustez ante cambios de amplitud y ruido.

## Conexión con el siguiente laboratorio
Usa la representación, operación o criterio de evaluación de este laboratorio como punto de partida del siguiente paso de la secuencia.
